# UC-WP-1 — External-ID Deduplication (IdentityMatcher Chain)

**Persona:** Data Pipeline Engineer

**Goal:** Demonstrate the first matcher in the IdentityMatcher chain — `external_id`.
Ingest a STAC item carrying an `external_id` property, re-submit the identical item,
and observe the configured write-policy behaviour:

- **REFUSE_RETURN** (default): duplicate returns `200` with the existing item (no DB write).
- **REFUSE_FAIL**: duplicate returns `409 Conflict`.

Confirm only one item exists after both submissions. Then demonstrate geohash fallback
when `external_id` is absent, and confirm a genuinely new item (different id, geometry,
and content) gets ingested with `201`.

**Use case:** UC-WP-1 — deduplication in nightly ETL pipelines

**Shipped:** commit `18190f7` — 2026-04-15

**Prerequisites:**
- `RoutingPluginConfig` with `IdentityMatcher` chain configured for the target collection
- PostgreSQL `stored_geohash` generated column + index present
- Env vars: `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`

In [ ]:
import json
import os
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
TOKEN = os.environ["DYNASTORE_TOKEN"]
CATALOG_ID = os.environ["CATALOG_ID"]
COLLECTION_ID = os.environ.get("COLLECTION_ID", "asis_dekadal")

ITEMS_URL = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)

# Unique run suffix so each notebook execution starts clean
RUN_ID = uuid.uuid4().hex[:8]
EXTERNAL_ID = f"asis-drought-2024-01-D1-{RUN_ID}"
ITEM_ID = f"test-ext-id-{RUN_ID}"

print(f"BASE_URL     : {BASE_URL}")
print(f"CATALOG_ID   : {CATALOG_ID}")
print(f"COLLECTION_ID: {COLLECTION_ID}")
print(f"EXTERNAL_ID  : {EXTERNAL_ID}")
print(f"ITEM_ID      : {ITEM_ID}")

## Step 1 — Initial ingest

POST a STAC item with `external_id` set in `properties`. The platform stores this value
and indexes it for fast duplicate detection. A `201 Created` response confirms the item
was written to the database for the first time.

In [ ]:
item_payload = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": ITEM_ID,
    "collection": COLLECTION_ID,
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [33.0, 3.0],
            [48.0, 3.0],
            [48.0, 15.0],
            [33.0, 15.0],
            [33.0, 3.0],
        ]],
    },
    "bbox": [33.0, 3.0, 48.0, 15.0],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        "external_id": EXTERNAL_ID,
        "eo:cloud_cover": 12.5,
        "title": "ASIS Dekadal Jan 2024 D1 (Ethiopia)",
    },
    "links": [],
    "assets": {},
}

resp = client.post(ITEMS_URL, content=json.dumps(item_payload))
print(resp.status_code, resp.text[:400])
assert resp.status_code == 201, f"Expected 201 on initial ingest, got {resp.status_code}: {resp.text}"

created = resp.json()
assert "id" in created, "Response body must contain 'id'"
item_id = created["id"]
print(f"\nItem created: id={item_id}")

## Step 2 — Re-submit same item (REFUSE_RETURN mode)

POST the identical payload a second time. With `REFUSE_RETURN` policy the platform
detects the `external_id` match, skips the DB write, and returns `200` with the
existing item. No duplicate row is created.

Both `200` and `201` are accepted here because the exact status code depends on whether
the collection is configured with `REFUSE_RETURN` or `REFUSE_FAIL`. The count assertion
in the next cell is the definitive deduplication check.

In [ ]:
resp = client.post(ITEMS_URL, content=json.dumps(item_payload))
print(resp.status_code, resp.text[:400])

# Accept either REFUSE_RETURN (200) or REFUSE_FAIL (409)
assert resp.status_code in (200, 201, 409), (
    f"Expected 200 (REFUSE_RETURN) or 409 (REFUSE_FAIL) on duplicate, "
    f"got {resp.status_code}: {resp.text}"
)

if resp.status_code in (200, 201):
    returned = resp.json()
    assert "id" in returned, "REFUSE_RETURN response must contain 'id'"
    print(f"REFUSE_RETURN: existing item returned (id={returned['id']})")
else:
    print("REFUSE_FAIL: duplicate rejected with 409")

In [ ]:
# Confirm deduplication: the collection must contain exactly one item matching EXTERNAL_ID
resp = client.get(ITEMS_URL, params={"filter": f"properties.external_id='{EXTERNAL_ID}'"})

if resp.status_code == 200:
    data = resp.json()
    matched = data.get("features", data.get("items", []))
    print(f"Items matching external_id: {len(matched)}")
    assert len(matched) == 1, (
        f"Expected exactly 1 item with external_id={EXTERNAL_ID!r}, found {len(matched)}"
    )
    print("Deduplication confirmed: exactly 1 item exists after 2 submissions.")
else:
    # Fallback: fetch by id directly and confirm it exists
    resp2 = client.get(f"{ITEMS_URL}/{item_id}")
    assert resp2.status_code == 200, f"Item {item_id} must be retrievable; got {resp2.status_code}"
    print(f"Item {item_id} confirmed present (filter query not supported — using GET by id).")

## Step 3 — REFUSE_FAIL mode demonstration

The exact HTTP status code for a duplicate depends on the collection's write-policy
configuration. Both behaviours are shown below with conditional assertions so the
notebook is valid regardless of which policy is active.

In [ ]:
resp = client.post(ITEMS_URL, content=json.dumps(item_payload))
print(f"Status: {resp.status_code}")
print(f"Body  : {resp.text[:300]}")

# Behavior depends on policy configuration
if resp.status_code == 409:
    detail = resp.json().get("detail", "")
    assert "already exists" in detail.lower() or resp.status_code == 409, (
        f"REFUSE_FAIL 409 detail should mention 'already exists'; got: {detail!r}"
    )
    print("REFUSE_FAIL: duplicate rejected with 409 — error detail confirmed.")
elif resp.status_code in (200, 201):
    returned = resp.json()
    assert "id" in returned, "REFUSE_RETURN response must contain 'id'"
    print("REFUSE_RETURN: existing item returned — no duplicate written.")
else:
    raise AssertionError(
        f"Unexpected status {resp.status_code} for duplicate submission: {resp.text}"
    )

## Step 4 — Geohash fallback (no external_id)

When `external_id` is absent from `properties`, the IdentityMatcher chain falls back
to the `geohash` matcher. Items with the same geometry produce the same stored geohash,
so re-submitting the same shape at the same precision triggers deduplication even without
an explicit external identifier.

In [ ]:
GEO_ITEM_ID = f"test-geo-{RUN_ID}"

geo_item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": GEO_ITEM_ID,
    "collection": COLLECTION_ID,
    # Identical geometry → identical geohash
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [38.0, 8.0],
            [42.0, 8.0],
            [42.0, 12.0],
            [38.0, 12.0],
            [38.0, 8.0],
        ]],
    },
    "bbox": [38.0, 8.0, 42.0, 12.0],
    "properties": {
        "datetime": "2024-01-10T00:00:00Z",
        # No external_id — geohash matcher must activate
        "title": "ASIS Geohash Test Item",
    },
    "links": [],
    "assets": {},
}

# First submission → 201
resp = client.post(ITEMS_URL, content=json.dumps(geo_item))
print(f"First submission: {resp.status_code}")
assert resp.status_code == 201, (
    f"Expected 201 on first geohash item submission, got {resp.status_code}: {resp.text}"
)
geo_created_id = resp.json().get("id")
print(f"Geohash item created: id={geo_created_id}")

# Second submission (same geometry) → dedup via geohash
resp2 = client.post(ITEMS_URL, content=json.dumps(geo_item))
print(f"Second submission (geohash match): {resp2.status_code}")
assert resp2.status_code in (200, 201, 409), (
    f"Expected 200/409 on geohash duplicate, got {resp2.status_code}: {resp2.text}"
)
if resp2.status_code in (200, 201):
    print("REFUSE_RETURN: geohash duplicate returned existing item.")
else:
    print("REFUSE_FAIL: geohash duplicate rejected with 409.")

## Edge cases

### All matchers miss → 201 (genuinely new item)

When `external_id`, geometry, and content all differ, no matcher fires and the item is
inserted as a new record. A `201` response confirms no false-positive deduplication.

In [ ]:
NEW_ITEM_ID = f"test-new-{RUN_ID}"

new_item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": NEW_ITEM_ID,
    "collection": COLLECTION_ID,
    # Different geometry → different geohash
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [-10.0, 10.0],
            [-5.0, 10.0],
            [-5.0, 15.0],
            [-10.0, 15.0],
            [-10.0, 10.0],
        ]],
    },
    "bbox": [-10.0, 10.0, -5.0, 15.0],
    "properties": {
        "datetime": "2024-06-15T00:00:00Z",
        # Different external_id → no match
        "external_id": f"completely-different-{uuid.uuid4().hex}",
        # Different content → content_hash matcher also misses
        "eo:cloud_cover": 77.3,
        "title": "Sahel Rainfall Jun 2024",
    },
    "links": [],
    "assets": {},
}

resp = client.post(ITEMS_URL, content=json.dumps(new_item))
print(resp.status_code, resp.text[:300])
assert resp.status_code == 201, (
    f"Expected 201 for genuinely new item (all matchers miss), got {resp.status_code}: {resp.text}"
)
new_id = resp.json().get("id")
print(f"Genuinely new item created: id={new_id}")

## Teardown

Delete all items created during this notebook run. A `204 No Content` response confirms
each deletion. Items that were not created (e.g., due to a failed step) are skipped
gracefully.

In [ ]:
to_delete = [
    ("external_id item", item_id),
    ("geohash item", geo_created_id),
    ("new item", new_id),
]

for label, del_id in to_delete:
    if not del_id:
        print(f"  Skipping {label} — id not available")
        continue
    resp = client.delete(f"{ITEMS_URL}/{del_id}")
    print(f"  DELETE {label} ({del_id}): {resp.status_code}")
    assert resp.status_code in (204, 404), (
        f"Expected 204 or 404 when deleting {label}, got {resp.status_code}: {resp.text}"
    )

print("Teardown complete.")
client.close()